In [ ]:
import pandas
from minio import Minio
import json
import pendulum

In [ ]:
minio_client = Minio(
    "localhost:9000",
    access_key="root",
    secret_key="password",
    secure=False
)

In [44]:
bucket_name = "bga-log-responses-test1"
objects = list(minio_client.list_objects(bucket_name))
dataframes = []

In [45]:
for obj in sorted(objects, key=lambda x: x.object_name):
    response = minio_client.get_object(bucket_name, obj.object_name)
    content = response.read()
    json_data: dict = json.loads(content)
    json_data.update(json_data["content"])
    if "results" in json_data:
        json_data.update({"data": json_data["results"]})
        del json_data["results"]
    if "error" in json_data:
        minio_client.remove_object(bucket_name, obj.object_name)
    if "data" in json_data:
        if json_data["data"] == "ok":
            minio_client.remove_object(bucket_name, obj.object_name)
        if "data" in json_data["data"]:
            if not json_data["data"]["data"]:
                minio_client.remove_object(bucket_name, obj.object_name)
    del json_data["content"]
    json_data["file_time"] = pendulum.parse(obj.object_name.replace(".json", ""))
    df = pandas.DataFrame([json_data])
    dataframes.append(df)

In [46]:
combined_df = pandas.concat(dataframes).reset_index(drop=True)
combined_df.sort_values(by="file_time", ascending=True).tail(100)

,status_code,status,data,file_time
0,200,1,"{'type': 'table', 'id': '565243417', 'history'...",2024-09-20 20:43:57.659760+00:00
1,200,1,"{'valid': 1, 'data': [{'channel': '/table/t565...",2024-09-20 20:43:57.818244+00:00
2,200,1,"{'type': 'table', 'id': '565243417', 'history'...",2024-09-20 20:43:58.389067+00:00
3,200,1,"{'valid': 1, 'data': [{'channel': '/table/t565...",2024-09-20 20:43:58.544111+00:00
4,200,1,"{'type': 'table', 'id': '565243417', 'history'...",2024-09-20 20:43:58.866588+00:00
5,200,1,"{'exists': True, 'translated': True, 'content'...",2024-09-20 20:43:58.895468+00:00
6,200,1,"{'valid': 1, 'data': [{'channel': '/table/t565...",2024-09-20 20:43:59.066728+00:00
7,200,1,"{'id': '565243417', 'game_id': '36', 'status':...",2024-09-20 21:31:46.251407+00:00
8,200,1,"{'id': '565243417', 'game_id': '36', 'status':...",2024-09-20 21:31:46.285434+00:00
9,200,1,"{'id': '565243417', 'game_id': '36', 'status':...",2024-09-20 21:31:46.295904+00:00
